In [ ]:
# Instala as bibliotecas necessárias
!pip install rapidfuzz scikit-learn pandas python-Levenshtein nltk


In [ ]:
# Importações de bibliotecas para análise e manipulação de dados
import pandas as pd
import unicodedata
import nltk
import matplotlib.pyplot as plt
from rapidfuzz import process, fuzz # Para fuzzy matching rápido
from Levenshtein import distance as levenshtein_distance # Para medir a similaridade entre as strings
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
# Transferirr o tokenizer do NLTK (necessário para tokenização)
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

Carregamento e preparação do dataset

In [ ]:
# Carregar o dataset
df = pd.read_csv('pokemon.csv')

# Visualizar as primeiras linhas
df.head()


,abilities,against_bug,against_dark,against_dragon,against_electric,against_fairy,against_fight,against_fire,against_flying,against_ghost,...,percentage_male,pokedex_number,sp_attack,sp_defense,speed,type1,type2,weight_kg,generation,is_legendary
0,"['Overgrow', 'Chlorophyll']",1.0,1.0,1.0,0.5,0.5,0.5,2.0,2.0,1.0,...,88.1,1,65,65,45,grass,poison,6.9,1,0
1,"['Overgrow', 'Chlorophyll']",1.0,1.0,1.0,0.5,0.5,0.5,2.0,2.0,1.0,...,88.1,2,80,80,60,grass,poison,13.0,1,0
2,"['Overgrow', 'Chlorophyll']",1.0,1.0,1.0,0.5,0.5,0.5,2.0,2.0,1.0,...,88.1,3,122,120,80,grass,poison,100.0,1,0
3,"['Blaze', 'Solar Power']",0.5,1.0,1.0,1.0,0.5,1.0,0.5,1.0,1.0,...,88.1,4,60,50,65,fire,NaN,8.5,1,0
4,"['Blaze', 'Solar Power']",0.5,1.0,1.0,1.0,0.5,1.0,0.5,1.0,1.0,...,88.1,5,80,65,80,fire,NaN,19.0,1,0


In [ ]:
# Selecionar apenas as colunas relevantes
cols = ['name', 'type1', 'type2', 'height_m', 'hp', 'attack', 'defense', 'speed', 'weight_kg','base_total']
df = df[cols]

# Conferir as colunas e dados restantes
df.head()


,name,type1,type2,height_m,hp,attack,defense,speed,weight_kg,base_total
0,Bulbasaur,grass,poison,0.7,45,49,49,45,6.9,318
1,Ivysaur,grass,poison,1.0,60,62,63,60,13.0,405
2,Venusaur,grass,poison,2.0,80,100,123,80,100.0,625
3,Charmander,fire,NaN,0.6,39,52,43,65,8.5,309
4,Charmeleon,fire,NaN,1.1,58,64,58,80,19.0,405


In [ ]:
# Verificar valores nulos antes da limpeza
print("Valores nulos antes:")
print(df.isnull().sum())

# Preencher valores ausentes de 'type2' com 'none'
df['type2'] = df['type2'].fillna('none')

# Remover linhas com valores nulos em 'height_m' ou 'weight_kg'
df = df.dropna(subset=['height_m', 'weight_kg'])

# Verificar novamente os valores nulos
print("Valores nulos depois:")
print(df.isnull().sum())


Valores nulos antes:
name            0
type1           0
type2         384
height_m       20
hp              0
attack          0
defense         0
speed           0
weight_kg      20
base_total      0
dtype: int64
Valores nulos depois:
name          0
type1         0
type2         0
height_m      0
hp            0
attack        0
defense       0
speed         0
weight_kg     0
base_total    0
dtype: int64


In [ ]:
# Garantir que colunas numéricas estão com os tipos corretos
numeric_cols = ['height_m', 'hp', 'attack', 'defense', 'speed', 'weight_kg', 'base_total']
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')

# Verificar tipos de dados e primeiras linhas
print(df.dtypes)
df[numeric_cols].head()

# Estatísticas descritivas para as colunas numéricas
df.describe()


name           object
type1          object
type2          object
height_m      float64
hp              int64
attack          int64
defense         int64
speed           int64
weight_kg     float64
base_total      int64
dtype: object


,height_m,hp,attack,defense,speed,weight_kg,base_total
count,781.000000,781.000000,781.000000,781.000000,781.000000,781.000000,781.000000
mean,1.163892,69.148528,77.693982,73.002561,66.279129,61.378105,428.774648
std,1.080326,26.621679,32.223734,30.806010,28.904200,109.354766,119.466916
min,0.100000,1.000000,5.000000,5.000000,5.000000,0.100000,180.000000
25%,0.600000,50.000000,55.000000,50.000000,45.000000,9.000000,323.000000
50%,1.000000,65.000000,75.000000,70.000000,65.000000,27.300000,440.000000
75%,1.500000,80.000000,100.000000,90.000000,85.000000,64.800000,508.000000
max,14.500000,255.000000,185.000000,230.000000,180.000000,999.900000,780.000000


In [ ]:
# Remove acentos e converte para minúsculas
def remove_acentos(texto: str) -> str:
    return ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    ).lower()

# Tokeniza o texto e gera bigramas (pares de palavras consecutivas)
def tokenize_and_generate_ngrams(text):
    tokens = text.split()
    bigrams = [tokens[i] + ' ' + tokens[i + 1] for i in range(len(tokens) - 1)]
    return tokens + bigrams

# Faz correspondência entre tokens e termos com base na distância de Levenshtein
def match_with_levenshtein(tokens, terms, max_dist=2):
    matched = []
    for token in tokens:
        for term in terms:
            if levenshtein_distance(token, term) <= max_dist:
                matched.append(term)
    return matched


Função Principal da busca de Pokémons com base na descrição

In [ ]:
def search_pokemon(description, df):
    # Pré-processamento da descrição
    description = remove_acentos(description.lower())
    tokens = tokenize_and_generate_ngrams(description)

    # Palavras associadas a cada atributo numérico
    all_terms = {
        'height_m': ['alto', 'baixo'],
        'hp': ['vida alta', 'vida baixa'],
        'attack': ['ataque alto', 'ataque baixo'],
        'defense': ['defesa alta', 'defesa baixa'],
        'speed': ['rapido', 'lento'],
        'weight_kg': ['pesado', 'leve']
    }

    # Normaliza os tipos únicos dos Pokémon (tipo1 e tipo2)
    types = df[['type1', 'type2']].stack().unique()
    normalized_types = [remove_acentos(t) for t in types]
    normalized_tokens = [remove_acentos(tk) for tk in tokens]

    # Matching mais rigoroso com fuzz.ratio (+ 90 para evitar confusões)
    matched_types = []
    for token in normalized_tokens:
        for t in normalized_types:
            if fuzz.ratio(token, t) >= 90:
                matched_types.append(t)

    # Remover duplicados
    matched_types = list(set(matched_types))

    # Inicializar filtros e critérios
    score_possible = 0
    filters = []
    sort_criteria = []
    matched_terms = []

    # Detectar termos qualitativos na descrição
    for terms in all_terms.values():
        matched_terms.extend(match_with_levenshtein(tokens, terms))

    # Função auxiliar para aplicar filtros com base em termos "alto" ou "baixo"
    def check_numeric_filter(column, high_label, low_label):
        nonlocal score_possible
        if high_label in matched_terms:
            score_possible += 1
            return (df[column] >= df[column].quantile(0.75), column, False)  # ordenação desc
        elif low_label in matched_terms:
            score_possible += 1
            return (df[column] <= df[column].quantile(0.25), column, True)   # ordenação asc
        return (None, None, None)

    # Aplicar os filtros detectados a partir da descrição
    for column, (high_label, low_label) in all_terms.items():
        filter_, sort_col, asc = check_numeric_filter(column, high_label, low_label)
        if filter_ is not None:
            filters.append((column, filter_))
            sort_criteria.append((sort_col, asc))

    # Iniciar dataframe filtrado
    filtered_df = df.copy()
    type_score = 0

    # Aplicar filtro por tipo (type1 ou type2)
    if matched_types:
        filtered_df = filtered_df[
            (filtered_df['type1'].apply(remove_acentos).isin(matched_types)) |
            (filtered_df['type2'].apply(remove_acentos).isin(matched_types))
        ]
        score_possible += 2
        type_score = 2

    # Função de cálculo de score baseado em filtros atendidos
    def calculate_score(row):
        score = type_score
        for column, filter_ in filters:
            if filter_.loc[row.name]:  # verifica se a linha passa no filtro
                score += 1
        return round((score / score_possible) * 100) if score_possible else 0

    # Aplicar pontuação e ordenação
    filtered_df['score'] = filtered_df.apply(calculate_score, axis=1)

    # Aplicar ordenação adicional pelos critérios definidos (ex: ataque alto, peso baixo, etc.)
    for sort_col, asc in reversed(sort_criteria):
        if sort_col:
            filtered_df = filtered_df.sort_values(by=sort_col, ascending=asc)

    # Ordenar por score final (relevância da correspondência com a descrição)
    filtered_df = filtered_df.sort_values(by='score', ascending=False)

    # Renomear coluna para apresentação
    filtered_df.rename(columns={'score': 'score (%)'}, inplace=True)

    # Retornar apenas os 5 primeiros resultados com as colunas mais relevantes
    return filtered_df[['name', 'type1', 'type2', 'height_m', 'hp', 'attack', 'defense', 'speed', 'weight_kg','base_total','score (%)']].head(5)

Execução da pesquisa

In [ ]:
print("=== Pesquisa de Pokémon por descrição ===")
descricao = input("Descreva o Pokémon (ex: rápido, pesado, ataque alto, tipo fogo...): ")
resultados = search_pokemon(descricao, df)
print(resultados.to_string(index=False))

=== Pesquisa de Pokémon por descrição ===
Descreva o Pokémon (ex: rápido, pesado, ataque alto, tipo fogo...): type fire pesado ataque alto
    name  type1  type2  height_m  hp  attack  defense  speed  weight_kg  base_total  score (%)
   Ho-Oh   fire flying       3.8 106     130       90     90      199.0         680        100
Reshiram dragon   fire       3.2 100     120      100     90      330.0         680        100
   Entei   fire   none       2.1 115     115       85    100      198.0         580        100
Camerupt   fire ground       1.9  70     120      100     20      220.0         560        100
Arcanine   fire   none       1.9  90     110       80     95      155.0         555        100
